Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 01 · Universe and identifiers

**Question:** Which securities and companies are being studied?

The September 30 IVV holdings, SEC ticker-to-CIK mapping and dated classification sources define the fixed research population. One security line is not always one company.

Research report · Method and code guide · Evidence map

**Collection record:** original execution outputs are retained. Read this notebook for provenance; use the offline reproduction command to rebuild the research without refreshing downloads.


**Chapter route**

Measurement and interpretation principles · Read the population · Read and run this notebook · Calculation source: foundation.same security · Results, examples and interpretation · normalize weights · attach classifications · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

An issuer and a security are different observational units. Normalize equity market values as $w_i=MV_i/\sum_j MV_j$; a portfolio weight measures fund allocation, not company size. Keep both share classes for security and portfolio work, while fixing one largest-snapshot-weight representative per issuer for company regressions.


## Read the population

Filter equity lines before identifying issuers; retain the raw source separately.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version. The original collection cells below remain archival; do not rerun them merely to read the calculations.


## Calculation source: foundation.same security

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Same-security rules: where a ticker's downloaded history is, or is not, the security held on 30 Sep 2026.

Decided by Sai on 13 Oct 2026 and first written in notebooks/02_prices.ipynb: recover the same security where
feasible; otherwise exclude transparently, with no substitute company and no invented history. One row per
exception, with the evidence behind it. This module is now the single place the rules are written down; the
build writes them to data/processed/same_security_rules_2026-09-30.csv.

Columns
- history_start: first session that belongs to this security (earlier prices are a predecessor or pre-listing).
- history_end: last session kept (later prices are dropped).
- price_source: "yahoo" (the main pull), "yahoo:<symbol>" or "fmp:<symbol>" (a recovered history from the supplement file).
- market_cap_valid_from: first date on which the vendor market cap uses the right share basis.
- evidence: the facts behind the decision.
"""

import pandas as pd

same_security_rule_records = [
    {"ticker": "WBD", "history_start": None, "history_end": "2026-09-30", "price_source": "fmp:WBD", "market_cap_valid_from": "2022-04-11",
     "evidence": "One SEC registrant since 2008 (CIK 1437107: Discovery Communications, Discovery Inc., Warner Bros. Discovery); DISCA renamed WBD "
                 "on 11 Apr 2022 with no price break. Yahoo holds a single bar (5 Oct 2026), which FMP's close matches; FMP adjusted returns match "
                 "Yahoo's to 0.4bp a day on 25 test names. Kept through the snapshot date as Sai instructed. FMP market cap before the rename uses "
                 "the post-merger share count (180-335% above SEC filings) and is rejected."},
    {"ticker": "PSKY", "history_start": "2025-08-07", "history_end": None, "price_source": "yahoo:SKYD", "market_cap_valid_from": None,
     "evidence": "SEC CIK 2041610 became Paramount Skydance Corp on 7 Aug 2025 and Skydance Corp on 6 Oct 2026; FMP records PSKY renamed SKYD on "
                 "6 Oct 2026. Yahoo and FMP agree exactly from 7 Aug 2025. Earlier SKYD prices are Paramount Global's and are excluded."},
    {"ticker": "SW", "history_start": "2024-07-08", "history_end": None, "price_source": "yahoo", "market_cap_valid_from": None,
     "evidence": "New registrant (CIK 2005951, created Feb 2024); NYSE registration (8-A12B) filed 5 Jul 2024; last zero-volume day 5 Jul 2024. "
                 "Earlier Yahoo prices are not this security."},
    {"ticker": "AMCR", "history_start": "2019-06-11", "history_end": None, "price_source": "yahoo", "market_cap_valid_from": None,
     "evidence": "New registrant Amcor plc (CIK 1748790); NYSE registration filed 7 Jun 2019; last zero-volume day 10 Jun 2019. "
                 "Earlier Yahoo prices are not this security."},
    {"ticker": "FERG", "history_start": "2021-03-08", "history_end": None, "price_source": "yahoo", "market_cap_valid_from": None,
     "evidence": "Ferguson plc NYSE trading from 8 Mar 2021, exchanged 1:1 into Ferguson Enterprises (CIK 2011641) on 1 Aug 2024 under an SEC "
                 "successor filing (8-K12B), same ticker, no price break. Sai chose to keep the plc NYSE years; earlier prices are excluded."},
    {"ticker": "FOX", "history_start": "2019-03-19", "history_end": None, "price_source": "yahoo", "market_cap_valid_from": None,
     "evidence": "Fox Corp (CIK 1754301) regular-way trading from 19 Mar 2019; 12-18 Mar was when-issued. The symbol belonged to "
                 "21st Century Fox before March 2019. Sai chose regular-way."},
    {"ticker": "FOXA", "history_start": "2019-03-19", "history_end": None, "price_source": "yahoo", "market_cap_valid_from": None,
     "evidence": "As FOX."},
]


**same_security_rules**

The same-security rules as a ticker-indexed frame, built the same way notebook 02 built it (so its CSV is unchanged).


In [2]:
def same_security_rules():
    """The same-security rules as a ticker-indexed frame, built the same way notebook 02 built it (so its CSV is unchanged)."""
    return pd.DataFrame(same_security_rule_records).set_index("ticker")  # a fresh frame on every call, so no caller can edit the shared rules


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [1]:
import pandas as pd
from pathlib import Path

ivv_holdings_csv_path = Path("../data/raw/ivv_holdings_2026-09-30.csv")  # the notebook runs from notebooks/, so data/ is one level up

ivv_holdings_all_lines = pd.read_csv(
    ivv_holdings_csv_path,
    skiprows=9,      # the first nine lines are the fund preamble (name, as-of date, shares outstanding), not holdings
    thousands=",",   # numbers are printed as "12,336,797,373.72"; without this they load as text
)

ivv_holdings_equity_only = ivv_holdings_all_lines[ivv_holdings_all_lines["Asset Class"] == "Equity"]  # drops cash, money market, collateral and the future

ivv_holdings_by_ticker = ivv_holdings_equity_only.set_index("Ticker")  # one row per security line, looked up by ticker

assert len(ivv_holdings_by_ticker) == 504                       # equity count recorded in the manifest
assert ivv_holdings_by_ticker.index.is_unique                   # a ticker index is only safe if no ticker repeats
assert ivv_holdings_by_ticker["Weight (%)"].dtype == "float64"  # thousands separators parsed, weights are numbers

ivv_holdings_by_ticker.head()

,Name,Sector,Asset Class,Market Value,Weight (%),Notional Value,Quantity,Price,Location,Exchange,Currency,FX Rate,Market Currency,Accrual Date
Ticker,,,,,,,,,,,,,,
NVDA,NVIDIA,Information Technology,Equity,7.364366e+10,8.37,7.364366e+10,322461078.0,228.38,United States,NASDAQ,USD,1.0,USD,-
AAPL,APPLE,Information Technology,Equity,6.476076e+10,7.36,6.476076e+10,194465078.0,333.02,United States,NASDAQ,USD,1.0,USD,-
MSFT,MICROSOFT,Information Technology,Equity,5.074848e+10,5.77,5.074848e+10,98944205.0,512.90,United States,NASDAQ,USD,1.0,USD,-
AMZN,AMAZON.COM INC,Consumer Discretionary,Equity,3.258656e+10,3.70,3.258656e+10,130790916.0,249.15,United States,NASDAQ,USD,1.0,USD,-
GOOGL,ALPHABET CLASS A,Communication,Equity,2.690395e+10,3.06,2.690395e+10,78190966.0,344.08,United States,NASDAQ,USD,1.0,USD,-


In [2]:
import json

sec_company_tickers_path = Path("../data/raw/sec_company_tickers_2026-10-09.json")  # SEC ticker-to-CIK file, see its manifest

sec_company_tickers_raw = json.loads(sec_company_tickers_path.read_text())                   # {"0": {"cik_str": 1045810, "ticker": "NVDA", "title": "NVIDIA CORP"}, ...}
sec_ticker_table = pd.DataFrame.from_dict(sec_company_tickers_raw, orient="index")            # one row per listed ticker: cik_str, ticker, title
sec_cik_by_ticker = sec_ticker_table.set_index("ticker")["cik_str"]                           # lookup series: ticker in, cik out
sec_issuer_name_by_ticker = sec_ticker_table.set_index("ticker")["title"]                     # lookup series: ticker in, registrant name out

ivv_holdings_ishares_ticker_as_column = ivv_holdings_by_ticker.reset_index().rename(columns={"Ticker": "ticker_ishares"})  # keep the iShares spelling as an ordinary column

ticker_yahoo_form = ivv_holdings_ishares_ticker_as_column["ticker_ishares"].str.replace(" ", "-")  # BRK B becomes BRK-B; Yahoo and the SEC file both use the hyphen
issuer_cik = ticker_yahoo_form.map(sec_cik_by_ticker).astype("Int64")                               # Int64 holds a missing cik as <NA> instead of turning the column into floats
issuer_name = ticker_yahoo_form.map(sec_issuer_name_by_ticker)

ivv_holdings_with_issuer = ivv_holdings_ishares_ticker_as_column.assign(
    ticker=ticker_yahoo_form,
    cik=issuer_cik,
    issuer_name=issuer_name,
).set_index("ticker")  # one row per security line, indexed by the project's ticker spelling

assert ivv_holdings_with_issuer.index.is_unique                                                   # the respelling must not merge two lines
assert set(ivv_holdings_with_issuer.index[ivv_holdings_with_issuer["cik"].isna()]) == {"HOLX"}    # only line the SEC no longer lists: unlisted, zero weight

issuers_with_more_than_one_line = ivv_holdings_with_issuer[ivv_holdings_with_issuer["cik"].duplicated(keep=False)]  # share classes of one issuer
issuers_with_more_than_one_line[["ticker_ishares", "Name", "cik", "issuer_name", "Weight (%)"]].sort_values("cik")

,ticker_ishares,Name,cik,issuer_name,Weight (%)
ticker,,,,,
NWSA,NWSA,NEWS CLASS A,1564708,NEWS CORP,0.02
NWS,NWS,NEWS CLASS B,1564708,NEWS CORP,0.01
GOOGL,GOOGL,ALPHABET CLASS A,1652044,Alphabet Inc.,3.06
GOOG,GOOG,ALPHABET CLASS C,1652044,Alphabet Inc.,2.45
FOXA,FOXA,FOX CLASS A,1754301,Fox Corp,0.02
FOX,FOX,FOX CLASS B,1754301,Fox Corp,0.01


In [3]:
columns_not_needed = [
    "ticker_ishares", "Asset Class", "Notional Value", "Quantity", "Location",
    "Exchange", "Currency", "FX Rate", "Market Currency", "Accrual Date",
]  # the raw file in data/raw still holds all of these

ivv_holdings_trimmed = ivv_holdings_with_issuer.drop(columns=columns_not_needed)

ivv_holdings_trimmed.head()

,Name,Sector,Market Value,Weight (%),Price,cik,issuer_name
ticker,,,,,,,
NVDA,NVIDIA,Information Technology,7.364366e+10,8.37,228.38,1045810,NVIDIA CORP
AAPL,APPLE,Information Technology,6.476076e+10,7.36,333.02,320193,Apple Inc.
MSFT,MICROSOFT,Information Technology,5.074848e+10,5.77,512.90,789019,MICROSOFT CORP
AMZN,AMAZON.COM INC,Consumer Discretionary,3.258656e+10,3.70,249.15,1018724,AMAZON COM INC
GOOGL,ALPHABET CLASS A,Communication,2.690395e+10,3.06,344.08,1652044,Alphabet Inc.


## normalize weights

`Market Value / total equity Market Value` avoids rounded printed weights. These are fund portfolio weights, not historical issuer sizes.


In [4]:
ivv_holdings_listed_only = ivv_holdings_trimmed.drop(index="HOLX")  # unlisted, zero weight, no cik, no prices to come

total_equity_market_value = ivv_holdings_listed_only["Market Value"].sum()                       # dollars in the 503 listed equity lines; cash and futures were dropped in the first cell
weight_from_market_value = ivv_holdings_listed_only["Market Value"] / total_equity_market_value  # exact share of equity dollars, free of the two-decimal rounding in Weight (%)

ivv_equity_holdings = ivv_holdings_listed_only.assign(weight=weight_from_market_value)  # weight is a decimal: the column sums to 1

assert abs(ivv_equity_holdings["weight"].sum() - 1) < 1e-12  # a full allocation
assert (ivv_equity_holdings["weight"] > 0).all()             # every remaining line is actually held
assert (ivv_equity_holdings["weight"] * 100 - ivv_equity_holdings["Weight (%)"]).abs().max() < 0.02  # matches the printed weights up to their rounding; a misread market value would break this

ivv_equity_holdings.sort_values("weight", ascending=False).head(10)

,Name,Sector,Market Value,Weight (%),Price,cik,issuer_name,weight
ticker,,,,,,,,
NVDA,NVIDIA,Information Technology,7.364366e+10,8.37,228.38,1045810,NVIDIA CORP,0.083829
AAPL,APPLE,Information Technology,6.476076e+10,7.36,333.02,320193,Apple Inc.,0.073718
MSFT,MICROSOFT,Information Technology,5.074848e+10,5.77,512.90,789019,MICROSOFT CORP,0.057767
AMZN,AMAZON.COM INC,Consumer Discretionary,3.258656e+10,3.70,249.15,1018724,AMAZON COM INC,0.037094
GOOGL,ALPHABET CLASS A,Communication,2.690395e+10,3.06,344.08,1652044,Alphabet Inc.,0.030625
AVGO,BROADCOM INC,Information Technology,2.226327e+10,2.53,351.19,1730168,Broadcom Inc.,0.025342
GOOG,ALPHABET CLASS C,Communication,2.158116e+10,2.45,340.74,1652044,Alphabet Inc.,0.024566
META,META PLATFORMS CLASS A,Communication,2.130769e+10,2.42,725.18,1326801,"Meta Platforms, Inc.",0.024255
MU,MICRON TECHNOLOGY,Information Technology,1.602920e+10,1.82,1065.11,723125,MICRON TECHNOLOGY INC,0.018246


In [5]:
import hashlib
import json
from datetime import datetime, timezone

import requests

gics_as_of_utc = "2026-09-30T23:59:59Z"  # take each Wikipedia page as it stood at the end of the holdings date
wikipedia_session = requests.Session()
wikipedia_session.headers["User-Agent"] = "quiet-index-500 research notebook (python-requests)"
raw_data_folder = Path("../data/raw")
repo_root_folder = Path("..").resolve()

def save_wikipedia_revision_as_of(page_title, file_stem):  # raw wikitext of the last revision at or before gics_as_of_utc, with a manifest
    wikitext_path = raw_data_folder / f"{file_stem}_as_of_2026-09-30.wikitext"
    manifest_path = wikitext_path.with_suffix(".manifest.json")
    if not manifest_path.exists():  # revisions are permanent, so the revision id makes the pull exactly repeatable
        revision_query = wikipedia_session.get("https://en.wikipedia.org/w/api.php", timeout=60, params={
            "action": "query", "prop": "revisions", "titles": page_title, "rvlimit": 1, "rvdir": "older",
            "rvstart": gics_as_of_utc, "rvprop": "ids|timestamp", "format": "json"})
        revision = list(revision_query.json()["query"]["pages"].values())[0]["revisions"][0]
        raw_response = wikipedia_session.get("https://en.wikipedia.org/w/index.php", params={"oldid": revision["revid"], "action": "raw"}, timeout=60)
        raw_response.raise_for_status()
        wikitext_path.write_bytes(raw_response.content)
        manifest = {
            "source": f"Wikipedia, '{page_title}', raw wikitext",
            "source_url": f"https://en.wikipedia.org/w/index.php?oldid={revision['revid']}&action=raw",
            "revision_id": revision["revid"],
            "revision_timestamp_utc": revision["timestamp"],
            "as_of_requested_utc": gics_as_of_utc,
            "pulled_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
            "file": wikitext_path.resolve().relative_to(repo_root_folder).as_posix(),
            "sha256": hashlib.sha256(raw_response.content).hexdigest(),
            "note": "Secondary source, CC BY-SA. Used for GICS sub-industry by company and the GICS structure; checked against the IVV sector.",
        }
        manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    return wikitext_path

sp500_list_wikitext_path = save_wikipedia_revision_as_of("List of S&P 500 companies", "wikipedia_sp500_list")
gics_structure_wikitext_path = save_wikipedia_revision_as_of("Global Industry Classification Standard", "wikipedia_gics_structure")

[json.loads(p.with_suffix(".manifest.json").read_text())["revision_timestamp_utc"] for p in [sp500_list_wikitext_path, gics_structure_wikitext_path]]

['2026-09-25T22:49:33Z', '2026-09-06T19:31:56Z']

## attach classifications

Ticker-to-subindustry joins reconstruct the hierarchy and check sector consistency. A correct September classification is still retrospective for earlier years.


In [6]:
import re

def strip_wiki_markup(cell_text):  # [[target|label]] becomes label, templates and html tags are dropped
    cell_text = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", cell_text)
    cell_text = re.sub(r"<[^>]+>|\{\{[^}]*\}\}", "", cell_text)
    return re.sub(r"\s+", " ", cell_text).strip()

def wikitext_row_cells(row_text):  # every cell in one table row, whether cells share a line ("|| a || b") or not
    cells = []
    for line in row_text.strip().splitlines():
        if not line.startswith("|") or line.startswith("|-"):
            continue
        cells += [cell.strip() for cell in line.lstrip("|").split("||")]
    return [re.sub(r'^rowspan="?\d+"?\s*\|\s*', "", cell) for cell in cells]

def first_wikitext_table(page_text, starting_at):  # the table text from a marker to its closing "|}"
    table_text = page_text[page_text.index(starting_at):]
    return table_text[:table_text.index("\n|}")]

# GICS structure: a code at each level (2, 4, 6, 8 digits) is followed by its name in the next cell
gics_structure_table = first_wikitext_table(gics_structure_wikitext_path.read_text(), "{|")
gics_name_by_code = {}
for row_text in gics_structure_table.split("\n|-"):
    row_cells = wikitext_row_cells(row_text)
    for position, cell in enumerate(row_cells[:-1]):
        if re.fullmatch(r"\d{2}|\d{4}|\d{6}|\d{8}", cell):
            gics_name_by_code.setdefault(cell, strip_wiki_markup(row_cells[position + 1]))
gics_codes_by_length = pd.Series(list(gics_name_by_code)).str.len().value_counts().to_dict()
assert gics_codes_by_length == {2: 11, 4: 25, 6: 74, 8: 163}  # sectors, industry groups, industries, sub-industries in the current GICS
sub_industry_code_by_name = {name: code for code, name in gics_name_by_code.items() if len(code) == 8}

# S&P 500 list: ticker -> GICS sub-industry name
sp500_list_table = first_wikitext_table(sp500_list_wikitext_path.read_text(), "! Security")
sub_industry_name_by_ticker = {}
for row_text in sp500_list_table.split("\n|-")[1:]:
    row_cells = wikitext_row_cells(row_text)
    ticker_in_wikipedia_form = re.search(r"\{\{[^|}]+\|([^}|]+)", row_cells[0]).group(1).strip()  # {{NyseSymbol|BRK.B}}
    sub_industry_name_by_ticker[ticker_in_wikipedia_form.replace(".", "-")] = strip_wiki_markup(row_cells[3])  # BRK.B becomes BRK-B

assert set(sub_industry_name_by_ticker) == set(ivv_equity_holdings.index)  # Wikipedia's 30 Sep membership equals the IVV snapshot

gics_sub_industry_code = ivv_equity_holdings.index.to_series().map(sub_industry_name_by_ticker).map(sub_industry_code_by_name)
assert gics_sub_industry_code.notna().all()  # every sub-industry name is in the structure

gics_columns = pd.DataFrame({
    "gics_sector_code": gics_sub_industry_code.str[:2],
    "gics_industry_group_code": gics_sub_industry_code.str[:4],
    "gics_industry_code": gics_sub_industry_code.str[:6],
    "gics_sub_industry_code": gics_sub_industry_code,
})
for level in ["sector", "industry_group", "industry", "sub_industry"]:
    gics_columns[f"gics_{level}"] = gics_columns[f"gics_{level}_code"].map(gics_name_by_code)
gics_columns = gics_columns[[f"gics_{level}{suffix}" for level in ["sector", "industry_group", "industry", "sub_industry"] for suffix in ["_code", ""]]]

ivv_sector_full_name = ivv_equity_holdings["Sector"].replace({"Communication": "Communication Services"})  # iShares shortens one label
assert (gics_columns["gics_sector"] == ivv_sector_full_name).all()  # the sector implied by each sub-industry matches the IVV sector, every name

ivv_equity_holdings_with_gics = ivv_equity_holdings.join(gics_columns)

gics_columns.nunique()

gics_sector_code             11
gics_sector                  11
gics_industry_group_code     25
gics_industry_group          25
gics_industry_code           69
gics_industry                69
gics_sub_industry_code      126
gics_sub_industry           126
dtype: int64

In [7]:
processed_data_folder = Path("../data/processed")  # tables the project derives, kept apart from untouched downloads in data/raw
processed_data_folder.mkdir(exist_ok=True)

ivv_equity_holdings_csv_path = processed_data_folder / "ivv_equity_holdings_2026-09-30.csv"
ivv_equity_holdings_with_gics.to_csv(ivv_equity_holdings_csv_path)  # the ticker index is written as the first column

ivv_equity_holdings_read_back = pd.read_csv(ivv_equity_holdings_csv_path, index_col="ticker")
assert ivv_equity_holdings_read_back.shape == ivv_equity_holdings_with_gics.shape                  # same rows and columns come back
assert (ivv_equity_holdings_read_back["weight"] - ivv_equity_holdings_with_gics["weight"]).abs().max() < 1e-15  # weights survive the trip to text and back

ivv_equity_holdings_csv_path

PosixPath('../data/processed/ivv_equity_holdings_2026-09-30.csv')

In [8]:
import hashlib
from datetime import datetime, timezone

repo_root_folder = Path("..").resolve()  # paths in manifests are written relative to the repo root, not to this notebook

def sha256_of_file(file_path):  # fingerprint: a 64-character string that changes if any byte of the file changes
    return hashlib.sha256(Path(file_path).read_bytes()).hexdigest()

def write_derived_table_manifest(table_path, input_paths, built_by, note=""):  # sidecar json recording what a table was built from
    table_path = Path(table_path)
    table_read_back = pd.read_csv(table_path)
    manifest = {
        "file": table_path.resolve().relative_to(repo_root_folder).as_posix(),
        "built_by": built_by,
        "built_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "inputs": [
            {"file": Path(input_path).resolve().relative_to(repo_root_folder).as_posix(), "sha256": sha256_of_file(input_path)}
            for input_path in input_paths
        ],
        "sha256": sha256_of_file(table_path),
        "bytes": table_path.stat().st_size,
        "rows": len(table_read_back),
        "columns": table_read_back.columns.tolist(),
        "note": note,
    }
    manifest_path = table_path.with_suffix(".manifest.json")
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    return manifest_path

ivv_equity_holdings_manifest_path = write_derived_table_manifest(
    ivv_equity_holdings_csv_path,
    input_paths=[ivv_holdings_csv_path, sec_company_tickers_path, sp500_list_wikitext_path, gics_structure_wikitext_path],
    built_by="notebooks/01_ivv_holdings.ipynb",
    note="IVV equity lines on 30 Sep 2026, HOLX dropped. weight = Market Value / total equity Market Value, decimal, sums to 1. "
         "Market Value is the fund's dollar holding, not the company's market cap. "
         "GICS industry group, industry and sub-industry come from Wikipedia as of 30 Sep 2026; their sector matches the IVV sector for every name.",
)

json.loads(ivv_equity_holdings_manifest_path.read_text())

{'file': 'data/processed/ivv_equity_holdings_2026-09-30.csv',
 'built_by': 'notebooks/01_ivv_holdings.ipynb',
 'built_at_utc': '2026-10-14T04:47:39+00:00',
 'inputs': [{'file': 'data/raw/ivv_holdings_2026-09-30.csv',
   'sha256': 'ab722f6637e678015a3052085c371eb4f29bd65f1c15c3b17b090d1edb099fe8'},
  {'file': 'data/raw/sec_company_tickers_2026-10-09.json',
   'sha256': 'a9595d33a34615553ee706c95458fff0bb9fec9642a9d61710f3e5dbd45ca23e'},
  {'file': 'data/raw/wikipedia_sp500_list_as_of_2026-09-30.wikitext',
   'sha256': '0f26e2860afab41cef5f4f7ef186fc9e07d402efe0bc5f0a8e206fbcc798b6f0'},
  {'file': 'data/raw/wikipedia_gics_structure_as_of_2026-09-30.wikitext',
   'sha256': 'd7171fae4dc5de8da33b14f64a6db1ecb908fc2d5cc37da0fce6e5a112b22f1e'}],
 'sha256': 'c64b3e43616769ebb9cdc497963f9dee6c53f1f8691d084becc27d0a1056c164',
 'bytes': 110530,
 'rows': 503,
 'columns': ['ticker',
  'Name',
  'Sector',
  'Market Value',
  'Weight (%)',
  'Price',
  'cik',
  'issuer_name',
  'weight',
  'gics_sect

In [9]:
universe_tickers_table = pd.DataFrame({"ticker": ivv_equity_holdings.index})  # the 503 tickers held constant from here on

universe_weights = ivv_equity_holdings[["weight", "Market Value"]].rename(columns={"Market Value": "market_value_usd"})  # ticker index, weight, fund dollar holding

sector_weights = (
    universe_weights.groupby(ivv_equity_holdings["Sector"]).sum()  # add up weight and dollars within each GICS sector
    .rename_axis("gics_sector")                                   # iShares prints "Communication" for Communication Services
    .sort_values("weight", ascending=False)
)

assert universe_tickers_table["ticker"].is_unique and len(universe_tickers_table) == 503
assert abs(sector_weights["weight"].sum() - 1) < 1e-12                                            # sectors partition the universe
assert abs(sector_weights["market_value_usd"].sum() - universe_weights["market_value_usd"].sum()) < 0.01  # no dollars lost in the grouping
assert len(sector_weights) == 11                                                                  # the eleven GICS sectors

universe_tickers_path = processed_data_folder / "universe_tickers_2026-09-30.csv"
universe_weights_path = processed_data_folder / "universe_weights_2026-09-30.csv"
sector_weights_path = processed_data_folder / "sector_weights_2026-09-30.csv"

universe_tickers_table.to_csv(universe_tickers_path, index=False)
universe_weights.to_csv(universe_weights_path)  # ticker index becomes the first column
sector_weights.to_csv(sector_weights_path)      # gics_sector index becomes the first column

snapshot_note = "Universe snapshot from IVV holdings dated 30 Sep 2026, an ETF proxy for the S&P 500. market_value_usd is the fund's holding, not company market cap."
for snapshot_path in [universe_tickers_path, universe_weights_path, sector_weights_path]:
    write_derived_table_manifest(snapshot_path, input_paths=[ivv_equity_holdings_csv_path],
                                 built_by="notebooks/01_ivv_holdings.ipynb", note=snapshot_note)

sector_weights

,weight,market_value_usd
gics_sector,,
Information Technology,0.397120,3.488681e+11
Financials,0.114008,1.001555e+11
Communication,0.099474,8.738742e+10
Health Care,0.092575,8.132708e+10
Consumer Discretionary,0.086009,7.555836e+10
Industrials,0.080465,7.068805e+10
Consumer Staples,0.043662,3.835658e+10
Energy,0.033963,2.983637e+10
Utilities,0.018778,1.649667e+10


## Findings

503 securities represent 500 issuers after removing the zero-weight unlisted HOLX line. Weights are normalized fund holding values, not issuer capitalization. September classifications and weights are later applied retrospectively.

## Analysis and takeaways

The 503-security / 500-issuer distinction determines every later denominator. Dated September 2026 weights and classifications make the historical analysis retrospective; identifier checks do not remove survivorship or hindsight.

## Bridge through the study

02 uses the tickers to collect histories; the shared foundation supplies identities, company representatives and weights to all research stages.
